# **Modelos NLP**

In [1]:
# [Config]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from tqdm import tqdm
from torch.utils.data import Dataset


import time
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cpu")
print("Usando dispositivo:", device)

Usando dispositivo: cpu


In [2]:
# [Data]
df = pd.read_csv('../Data/Clean_words.csv')
df = df.rename(columns={'macro_label': 'Category'})
print(f"Registros totales: {len(df)}")
df.head()


Registros totales: 2483


,Category,Resume_str,clean_text,tokens,bert_text,fasttext_text
0,Servicios Profesionales y Públicos,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,administrator marketing associate administrato...,"['administrator', 'marketing', 'associate', 'a...",hr administratormarketing associate\n...,__label__HR administrator marketing associate ...
1,Servicios Profesionales y Públicos,"HR SPECIALIST, US HR OPERATIONS ...",specialist operation summary versatile medium ...,"['specialist', 'operation', 'summary', 'versat...",hr specialist us hr operations ...,__label__HR specialist operation summary versa...
2,Servicios Profesionales y Públicos,HR DIRECTOR Summary Over 2...,director summary year experience recruiting pl...,"['director', 'summary', 'year', 'experience', ...",hr director summary over ...,__label__HR director summary year experience r...
3,Servicios Profesionales y Públicos,HR SPECIALIST Summary Dedica...,specialist summary dedicated driven dynamic ye...,"['specialist', 'summary', 'dedicated', 'driven...",hr specialist summary dedica...,__label__HR specialist summary dedicated drive...
4,Servicios Profesionales y Públicos,HR MANAGER Skill Highlights ...,manager skill highlight skill department start...,"['manager', 'skill', 'highlight', 'skill', 'de...",hr manager skill highlights ...,__label__HR manager skill highlight skill depa...


In [3]:
# 3. División estratificada
train_val, test = train_test_split(
    df, test_size=0.15, stratify=df['Category'], random_state=SEED
)
train, val = train_test_split(
    train_val, test_size=0.17647,  # 0.17647 * 0.85 ≈ 0.15 total
    stratify=train_val['Category'],
    random_state=SEED
)
print("División de grupos de datos")
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

# Codificar etiquetas
train['label'] = train['Category'].astype('category').cat.codes
val['label'] = val['Category'].astype('category').cat.codes
test['label'] = test['Category'].astype('category').cat.codes

num_labels = train['label'].nunique()
print("Número de clases:", num_labels)


División de grupos de datos
Train: 1737 | Val: 373 | Test: 373
Número de clases: 5


In [4]:
# [Def]

# Pesos de clase balanceados 
def get_class_weights(train_df, device):
    """Calcula pesos de clase balanceados"""
    # Calcula los pesos inversamente proporcionales a la frecuencia de cada clase
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_df['label']),
        y=train_df['label']
    )
    # Convierte los pesos a tensores de PyTorch
    return torch.tensor(class_weights, dtype=torch.float).to(device)

def evaluate_model(model, data_loader, device):
    """Evalúa el modelo y guarda reporte sklearn"""
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluando"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            preds.extend(torch.argmax(logits, axis=1).cpu().numpy())
            labels.extend(batch['labels'].cpu().numpy())
    report = classification_report(labels, preds, output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    df_report.to_csv(f"results/{model}_class_report.csv", index=True)
    return report


## DistilBERT

In [5]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset, DataLoader

In [6]:
# Etiquetado y selección de texto para BERT
train['text'] = train['bert_text']
val['text'] = val['bert_text']
test['text'] = test['bert_text']

In [7]:
# Cargar el tokenizer [sin distinguir entre Mayús-Minus] con long max de 512
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
MAX_LEN = 512

class BertDataset(Dataset):
    # Tokenizar textos, truncar, padding hasta max_length
    def __init__(self, df, tokenizer):
        self.encodings = tokenizer(
            df['text'].tolist(),
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN
        )
        self.labels = df['label'].tolist()
    # Diccionario de tensores
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BertDataset(train, tokenizer)
eval_dataset = BertDataset(val, tokenizer)

In [8]:
# Modelo y pesos
# Modelo para clasificacion de secuencias, con n clases.
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

class_weights = get_class_weights(train, device)    # Calcular pesos para balanceo
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights) # Función de perdida

from transformers import Trainer

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = loss_fn(outputs.logits, labels)  # loss_fn usa tus class_weights
        return (loss, outputs) if return_outputs else loss

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    logging_strategy="steps",
    logging_dir="./logs",
    use_cpu=True,
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

In [10]:
trainer.train()
trainer.state.log_history
log_df = pd.DataFrame(trainer.state.log_history)

val_loader = DataLoader(eval_dataset, batch_size=16)
evaluate_model(model, val_loader, device)

torch.save(model.state_dict(), "results/modelo_BERT.pt")

Epoch,Training Loss,Validation Loss


## BiLSTM [Word2Vec]

In [ ]:
# Etiquetado y selección de texto para Word2Vec
train['text'] = train['tokens']
val['text'] = val['tokens']
test['text'] = test['tokens']

## CNN-1D

In [ ]:
# Etiquetado y selección de texto para 
train['text'] = train['']
val['text'] = val['']
test['text'] = test['']

## TF-IDF [XGBoost]

In [ ]:
# Etiquetado y selección de texto para 
train['text'] = train['clean_text']
val['text'] = val['clean_text']
test['text'] = test['clean_text']

## FastTest

In [ ]:
# Etiquetado y selección de texto para 
train['text'] = train['fasttext_text']
val['text'] = val['fasttext_text']
test['text'] = test['fasttext_text']